In [ ]:
# Download the original book metadata dataset from Kaggle.
# This dataset serves as the starting point for the recommendation pipeline.

import kagglehub

# Download latest version
path = kagglehub.dataset_download("dylanjcastillo/7k-books-with-metadata")

print("Path to dataset files:", path)

In [ ]:

import pandas as pd
books = pd.read_csv(f"{path}/books.csv")

In [ ]:
books.describe()


In [ ]:
books.isnull().sum().to_frame(name='Missing')


In [ ]:
# Visualize the distribution of missing values across
# all features using a heatmap.

import matplotlib.pyplot as plt
import seaborn as sns
ax = plt.axes()
sns.heatmap(books.isna().transpose(), cbar=False, ax=ax)

plt.xlabel("columns")
plt.ylabel("missing values")
plt.imshow

In [ ]:
# Create additional features that may be useful during
# the exploratory analysis.
# missing_description indicates whether a description exists.

from datetime import datetime #NEWW
current_year = datetime.now().year

books["missing_description"] = ( books["description"].isnull().astype(int)) #NEWW
books["age_of_book"] = current_year - books["published_year"]

In [1]:
# Analyze the relationships between selected numerical
# features using Spearman correlation.
columns_of_interest = ["num_pages", "age_of_book", "missing_description", "average_rating"]
correlation_matrix = books[columns_of_interest].corr(method= "spearman")
sns.set_theme(style="white")
plt.figure(figsize=(10, 10))
heatmap = sns.heatmap(correlation_matrix, annot=True, fmt= ".2f", cmap="coolwarm", cbar_kws={"label": "spearman correlation"})
heatmap.set_title("correlation heatmap")
plt.show()

NameError: name 'books' is not defined

In [ ]:
# Remove records that do not contain the minimum amount
# of information required for the recommendation system.

required_columns = [
    "description",
    "num_pages",
    "average_rating",
    "published_year"
]

book_missing = books.dropna(subset=required_columns).copy()
book_missing["categories"].value_counts().reset_index().sort_values("count", ascending = False)
book_missing
books_missing_desc = books[books["description"].isna()]
books_missing_desc[["title","authors"]].to_csv("missing_books_description.csv", index=False)


In [ ]:
book_missing.iloc[6]


In [ ]:
# Calculate the number of words contained in each
# book description.
descriptionn_words = book_missing["description"].str.split() #NEWW
word_counts = descriptionn_words.str.len()

book_missing["words_in_descriptionn"] = word_counts
book_missing

In [ ]:
# Display descriptions within the selected length range
# for manual inspection.
book_missing.loc[book_missing["words_in_description"].between(1, 4), "description"]
book_missing.loc[book_missing["words_in_description"].between(15, 24), "description"]
book_missing.loc[book_missing["words_in_description"].between(24, 34), "description"]


In [ ]:
book_missing.loc[book_missing["words_in_description"].between(24, 34),["description"]]

In [ ]:
book_missing_25_words = book_missing[book_missing["words_in_description"] >=25].copy()


In [ ]:
# Combine title and subtitle into a single feature.
# If a subtitle is unavailable, the title is used alone.
book_missing_25_words["title_ad_subtitle"] = (np.where(book_missing_25_words["subtitle"].isna(), book_missing_25_words["title"],
             book_missing_25_words[["title", "subtitle"]].astype(str).agg(":".join, axis=1)))
book_missing_25_words[["title_ad_subtitle"]]

In [ ]:
# Create a tagged description by combining the ISBN
# with the book description. This identifier is later
# used during semantic retrieval.
book_missing_25_words["tagged_description"] = book_missing_25_words[["isbn13", "description"]].astype(str).agg(" ".join, axis=1)
book_missing_25_words

In [ ]:
(
    book_missing_25_words
    .drop(["subtitle", "missing_description", "age_of_book", "words_in_description"], axis=1)
    .to_csv("books_cleaned.csv", index = False)
)